# Quickstart

[`proto-tools`](https://github.com/evo-design/proto-tools) is an open source infrastructure layer,
that provides access to a large collection of computational biology and biological AI tools behind a
single, uniform Python interface. This guide covers how to set-up the package and run a tool.

## Installation

To begin, pip install the package in a virtual environment. The base environment
of proto-tools is intentionally light for maximum compatibility. The base package
requires Python 3.10 or later.

```bash
pip install git+https://github.com/evo-design/proto-tools.git
```

## Example tool call

All models and tools have their standard operations divided into functions
that operate over the following abstractions:

- `Inputs`: Contain the actual biological entities the tool operates over.
- `Configs`: Contain runtime settings that can be adjusted to control tool behavior.
- `Outputs`: The object that is produced by the tool.

<img src="assets/quickstart/call-pattern.svg" alt="Input plus Config feed into run_*(), which returns an Output" width="560">

These abstractions are [Pydantic](https://docs.pydantic.dev/) models, which declare each field with
a type and the values it is allowed to take. Pydantic enforces those declarations the moment the
object is constructed, so a value outside a permitted range is rejected immediately and names the
offending field, rather than failing part-way through a run that has already loaded a model onto a
GPU.

Let's show a quick example of how to call a tool. Here we will call Protenix to fold a protein complex. 

To begin, it may be helpful to look at our documentation page for Protenix, which contains a section on the Input, Config, and Output models and the various fields they define.


<a href="https://proto.evodesign.org/docs/tools/structure-prediction/protenix">
  <img src="https://proto-bio.github.io/proto-assets/images/tool/protenix/social.png" alt="Protenix — Structure Prediction" width="380">
</a>


On the documentation page above, we can see the run function and the various models it takes in as arguments. Let's import them below.

In [ ]:
from proto_tools import run_protenix, ProtenixInput, ProtenixConfig

# The input holds the biological entities. This is avGFP, the green fluorescent protein.
GFP = (
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQ"
    "HDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNG"
    "IKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"
)
inputs = ProtenixInput(complexes=[GFP])

# Every config field carries a default, so the config can be omitted entirely.
# Here we lower the number of diffusion samples to 1 to make the run quicker.
config = ProtenixConfig(num_diffusion_samples=1)

# Finally, we can run the function
output = run_protenix(inputs, config)

In [ ]:
# We can then take a look at the outputs we recieved back
structure = output.structures[0]

print(f"pLDDT:      {structure.metrics.avg_plddt:.3f}")
print(f"confidence: {structure.metrics.confidence_score:.3f}")

And that's it! All environment set-up is handled automatically for us under the hood. The next time this tool is called, inference will be subsantially faster as the environment and weights will already be set up on this machine.

## Go deeper

For the complete runtime reference, covering identifier resolution, the registry and CLI surfaces,
schema and documentation extraction, gated model weights, and calling conventions, consult the
developer notes in the proto-tools repository:

<a href="https://github.com/evo-design/proto-tools/blob/main/notes/finding-tools.md" class="go-deeper-card">
  <svg class="go-deeper-icon" width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M4 19.5A2.5 2.5 0 0 1 6.5 17H20"/><path d="M6.5 2H20v20H6.5A2.5 2.5 0 0 1 4 19.5v-15A2.5 2.5 0 0 1 6.5 2z"/></svg>
  <span class="go-deeper-body">
    <span class="go-deeper-title">Finding and Calling Tools</span>
    <span class="go-deeper-desc">Identifier resolution, registry methods, schemas, docs extraction, gated weights, and calling patterns.</span>
  </span>
</a>

## Next Steps

Continue with **[Tool Environments](https://proto.evodesign.org/docs/tools/guides/tool-environments)**,
the next guide in this series.

<CardGroup cols={2}>
  <Card title="Tool Environments" icon="box" href="/tools/guides/tool-environments">
    How a tool's isolated environment is built on first call and cached afterward.
  </Card>
  <Card title="Tool Persistence" icon="bolt" href="/tools/guides/tool-persistence">
    Keep a model loaded across calls to skip repeated load times.
  </Card>
  <Card title="Device Management" icon="microchip" href="/tools/guides/device-management">
    How tools are placed on GPUs, with LRU eviction and CPU offload.
  </Card>
  <Card title="Cloud Inference" icon="cloud" href="/tools/guides/cloud-inference">
    Dispatch tool runs to remote compute.
  </Card>
</CardGroup>